# RAG Q&A App — Phase 2
**Author:** Shruti  
**Description:** A complete Retrieval-Augmented Generation (RAG) pipeline built with LangChain, ChromaDB, and Groq. Answers questions about any document using semantic search and LLM generation.

**Stack:** Python, LangChain, ChromaDB, HuggingFace Embeddings, Groq API (LLaMA 3.1)

---

## Setup
Install libraries and import dependencies.

In [ ]:
!pip install langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface chromadb sentence-transformers groq -q

import json
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Replace with your Groq API key from console.groq.com
GROQ_API_KEY = "your-key-here"

llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant"
)

print("Setup complete!")

---
## Day 7 — Text Splitting & ChromaDB

**Skills practiced:** Splitting documents into chunks, converting to embeddings, storing in a vector database, querying by meaning.

### Exercise 1 — Split text into chunks
Create a text, split it into chunks using `RecursiveCharacterTextSplitter`, and print each chunk.

In [ ]:
text = """Machine Learning focuses on developing algorithms that allow computers to learn from data and improve their accuracy over time.
Generative AI uses complex models to create original content like realistic images, music, and professional-grade text.
Computer Vision enables machines to identify and process visual information from the world, much like human sight.
Natural Language Processing helps computers understand and interact with human speech and written language effectively.
AI Ethics examines the moral implications of technology, focusing on fairness, accountability, and user privacy."""

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_text(text)

for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")
    print("---")

### Exercise 2 — Convert chunks to embeddings
Use HuggingFace sentence-transformers to convert text chunks into numerical vectors.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(chunks)

print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding size: {len(embeddings[0])}")
print(f"First embedding (first 5 numbers): {embeddings[0][:5]}")

### Exercise 3 — Store in ChromaDB
Store chunks and their embeddings in ChromaDB for fast similarity search.

In [ ]:
import chromadb

chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="day7_knowledge")

collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Stored {collection.count()} chunks in ChromaDB")

### Exercise 4 — Query ChromaDB by meaning
Search ChromaDB with a question and retrieve the most relevant chunk.

In [ ]:
results = collection.query(
    query_texts=["How does AI handle ethical concerns?"],
    n_results=1
)

print("Most relevant chunk:")
print(results["documents"][0][0])

### Exercise 5 — Full RAG pipeline
Combine ChromaDB retrieval with the LLM to answer questions using document context.

In [ ]:
context = results["documents"][0][0]

response = llm.invoke([
    SystemMessage(content=f"You are a helpful assistant. Use this context: {context}"),
    HumanMessage(content="How does AI handle ethical concerns?")
])

print("RAG Answer:")
print(response.content)

---
## Day 8 — Better Chunking & Multiple Queries

**Skills practiced:** Larger chunk sizes for more context, querying multiple questions, comparing retrieved chunks.

### Exercises 1-3 — Create knowledge base, store and query with multiple questions
Use a richer document, larger chunks, and compare retrieval results for different questions.

In [ ]:
text = """
Machine Learning is a subset of AI that enables computers to learn from data without being explicitly programmed. It uses algorithms to find patterns in data and make predictions. Common types include supervised learning, unsupervised learning, and reinforcement learning.

Retrieval-Augmented Generation combines information retrieval with text generation. It first retrieves relevant documents from a knowledge base, then uses them as context for the LLM. RAG reduces hallucinations by grounding responses in real data.

Embeddings are numerical representations of text that capture semantic meaning. Similar texts produce similar embeddings. They are used in search, recommendation systems, and RAG pipelines to find relevant information.

Large Language Models are trained on vast amounts of text data to understand and generate human language. Examples include GPT-4, Claude, and LLaMA. They can perform tasks like summarization, translation, and question answering.

Vector databases store embeddings and enable fast similarity search. Examples include ChromaDB, Pinecone, and Weaviate. They are essential components of RAG systems for retrieving relevant context.
"""

with open("AI.txt", "w") as f:
    f.write(text)

with open("AI.txt", "r") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

chroma_client2 = chromadb.Client()
collection2 = chroma_client2.create_collection(name="day8_knowledge")
collection2.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print(f"Stored {collection2.count()} chunks")

# Query with two different questions
results1 = collection2.query(query_texts=["How do vector databases work?"], n_results=1)
results2 = collection2.query(query_texts=["What is supervised learning?"], n_results=1)

print("\nQ1 most relevant chunk:")
print(results1["documents"][0][0])
print("\nQ2 most relevant chunk:")
print(results2["documents"][0][0])

### Exercises 4-5 — Full RAG pipeline with better chunks
Pass retrieved context to LLM and get a grounded answer.

In [ ]:
context = results1["documents"][0][0]

response = llm.invoke([
    SystemMessage(content=f"You are a helpful assistant. Use this context: {context}"),
    HumanMessage(content="How do vector databases work?")
])

print("Answer:")
print(response.content)

---
## Day 9 — LangChain RAG Pipeline

**Skills practiced:** Using LangChain's `Chroma`, `HuggingFaceEmbeddings`, and `ChatGroq` to build a cleaner, more professional RAG pipeline.

### Exercises 1-5 — Full LangChain RAG pipeline
Replace manual ChromaDB and embedding code with LangChain abstractions.

In [ ]:
with open("AI.txt", "r") as f:
    text = f.read()

# Split
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

# Embed and store with LangChain
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="langchain_knowledge"
)

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Stored {vectorstore._collection.count()} chunks!")

# Loop through questions
questions = [
    "What is the difference between supervised and unsupervised learning?",
    "How do vector databases work?",
    "What are embeddings used for?"
]

results = []
for question in questions:
    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])
    response = llm.invoke([
        SystemMessage(content=f"You are a helpful assistant. Use this context: {context}"),
        HumanMessage(content=question)
    ])
    results.append({"question": question, "answer": response.content})
    print(f"Done: {question[:50]}...")

with open("day9_results.json", "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved {len(results)} results!")

---
## Day 10 — Interactive Q&A App

**Skills practiced:** Building an interactive Q&A system that answers any question about a document using RAG.

**This is the Phase 2 final project** — a complete, working RAG application.

### Step 1 — Create a richer document

In [ ]:
document = """
Artificial Intelligence (AI) is the simulation of human intelligence in machines.
AI systems are designed to perform tasks that typically require human intelligence,
such as visual perception, speech recognition, decision-making, and language translation.

Machine Learning is a subset of AI that gives systems the ability to learn from data.
Instead of being explicitly programmed, ML models identify patterns and make decisions.
Common algorithms include linear regression, decision trees, and neural networks.

Deep Learning is a subset of machine learning that uses neural networks with many layers.
These networks can automatically learn representations from raw data.
Deep learning powers applications like image recognition, natural language processing, and speech synthesis.

Natural Language Processing (NLP) enables computers to understand human language.
NLP techniques include tokenization, stemming, named entity recognition, and sentiment analysis.
Modern NLP is powered by transformer models like BERT and GPT.

Computer Vision enables machines to interpret visual information from images and videos.
Applications include facial recognition, object detection, medical image analysis, and autonomous vehicles.
Convolutional Neural Networks (CNNs) are the backbone of most computer vision systems.

Reinforcement Learning trains agents to make decisions by rewarding good behavior.
The agent learns through trial and error, maximizing cumulative rewards over time.
Applications include game playing, robotics, and recommendation systems.
"""

with open("ai_document.txt", "w") as f:
    f.write(document)

print("Document created!")

### Step 2 — Build the RAG pipeline

In [ ]:
with open("ai_document.txt", "r") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(text)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    collection_name="qa_app_knowledge"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print(f"Stored {vectorstore._collection.count()} chunks — Ready to answer questions!")

### Step 3 — Interactive Q&A loop
Ask any question about the document. Type `quit` to exit.

In [ ]:
while True:
    question = input("Ask a question about AI (or type 'quit' to exit): ")
    if question == "quit":
        print("Goodbye!")
        break

    docs = retriever.invoke(question)
    context = "\n".join([doc.page_content for doc in docs])

    response = llm.invoke([
        SystemMessage(content=f"You are a helpful assistant. Use this context to answer: {context}"),
        HumanMessage(content=question)
    ])

    print("\nAnswer:")
    print(response.content)
    print("\n" + "-"*50 + "\n")

---
## Phase 2 Summary

**What this project builds:**
- A complete RAG pipeline from scratch
- LangChain integration for cleaner code
- ChromaDB vector storage for semantic search
- Interactive Q&A app over any document

**Key concepts learned:**
- Text splitting and chunking strategies
- Embeddings and vector similarity search
- ChromaDB for storing and querying vectors
- LangChain abstractions for RAG
- Context-aware LLM prompting

**Next:** Phase 3 — Agents & Tool Use